---
title: Get started
description: How to use the Executor primitive in Qiskit Runtime.

---

# Get started with the Executor primitive

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

Similar to the [Sampler](/docs/guides/get-started-with-sampler) primitive, Executor samples output registers from quantum circuit executions, but it does not have any built in error suppression or mitigation. Instead, it's part of the [directed execution model](/docs/guides/directed-execution-model) that provides the ingredients to capture design intents on the client side, and shifts the costly generation of circuit variants to the server side. Executor follows the directives provided in circuit annotations and options, generates and binds parameter values, executes the bound circuits on the hardware, and returns the execution results and metadata. It does not make any implicit decisions for you and gives you full control and transparency.

<Admonition type="note">
The Qiskit package does not yet have a base class for the Executor primitive.
</Admonition>

## Steps to use the Executor primitive

### 1. Initialize the account

Because Qiskit Runtime is a managed service, you first need to initialize your account. You can then select the QPU you want to use to calculate the expectation value.

Follow the steps in the [Set up your IBM Cloud account](cloud-setup) if you don't already have an account.

In [11]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

In [12]:
print(backend)

<IBMBackend('ibm_boston')>



### 2. Create a circuit

You need at least one circuit to use the Executor primitive.  It can optionally have observables and paramaters.

In [13]:
from qiskit.circuit import QuantumCircuit

# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

# Using `measure_all` automatically creates the necessary 
# classical registers.
circuit.measure_all()

The circuit and observable need to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [14]:
from qiskit.transpiler import generate_preset_pass_manager

# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### 3. Initialize a `QuantumProgram`

Initialize a `QuantumProgram` with your workload. A `QuantumProgram` is made up of `QuantumProgramItems`. In general, each item consists of a circuit (possibly with parameter values or a `Samplex`) and a `shape` that determines executions. For full details, see [Executor inputs and outputs](/docs/guides/executor-input-output).

The following cell initializes a `QuantumProgram` and specifies to perform 25 shots for every configuration of each item in the program. Next, it appends the transpiled target circuit.

In [15]:
from qiskit_ibm_runtime.quantum_program import QuantumProgram

# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

### 4. Group gates and measurements into annotated boxes


In [16]:
from samplomatic.transpiler import generate_boxing_pass_manager

# Transpile the circuit, additionally grouping gates and measurements into annotated boxes
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
preset_pass_manager.post_scheduling = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
)
boxed_circuit = preset_pass_manager.run(circuit)

### 5. Optional: Build a template and samplex, and add them to the program

To specify a parameterized circuit without explicit parameters, you can use a _template_ and an optional _samplex item_.  A template is a circuit with parametric gates, but doesn't require that you specify fixed parameter values. A samplex can be used to produce randomized parameters on the server side.  Both of these items save memory by passing less data.

The `shape` request argument requests an extension of the implicit shape, determined by the parameters.  In particular, because this circuit has three parameters, parameter values require three numbers. Therefore, the shape is set to  `(3,)`. 

See the Samplomatic [API](https://github.com/Qiskit/samplomatic/) documentation for full details about `samplomatic.samplex.Samplex` and its arguments.

In [17]:
from samplomatic import build

# Build the template and the samplex
template, samplex = build(boxed_circuit)

# Append the template and samplex as a `samplex_item`
program.append_samplex_item(
    template,
    samplex=samplex,
    shape=(2,),
)

### 6. Invoke Executor and get results

Run the `QuantumProgram` on an IBM&reg; backend by using the `Executor` primitive with default options. See [Executor options](/docs/guides/executor-options) to learn about the available options.

In [18]:
from qiskit_ibm_runtime import Executor

# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job
# Retrieve the result
# result = job.result()

<RuntimeJobV2('d7jshcf16ugs73ev8asg', 'executor')>

In [19]:
result = job.result()

The result is of type [`QuantumProgramResult`](https://qiskit.github.io/qiskit-ibm-runtime/stubs/qiskit_ibm_runtime.quantum_program.QuantumProgramResult.html#qiskit_ibm_runtime.quantum_program.QuantumProgramResult). See [Executor input and output](/docs/guides/executor-input-output) to learn about the result object.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try some [Executor examples](/docs/guides/executor-examples).
    - Understand [Executor input and output](/docs/guides/executor-input-output).
    - Learn about [Executor broadcasting semantics](/docs/guides/executor-broadcasting).
</Admonition>